Checking GPU Requirements

In [3]:
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")


PyTorch Version: 2.14.1+cu130
GPU Detected: NVIDIA GeForce RTX 5060 Laptop GPU


Verify BitsAndBytes 4-Bit Quantization Compatibility

In [4]:
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("BitsAndBytes 4-bit configuration successfully initialized!")

/home/prathoseraajv/cross_lingual/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BitsAndBytes 4-bit configuration successfully initialized!


Load And Split Dataset

In [2]:
import json
import random
from datasets import Dataset, DatasetDict

dataset_path = "tanglish.jsonl"

# 1. Load raw dataset
with open(dataset_path, "r", encoding="utf-8") as f:
    raw_records = [json.loads(line) for line in f]

print(f"Loaded {len(raw_records)} records from {dataset_path}")

# 2. Format records into Gemma instruction template
def format_record(record):
    instruction = record.get("instruction", "").strip()
    inp = record.get("input", "").strip()
    output = record.get("output", "").strip()
    
    if inp:
        user_msg = f"{instruction}\n\nContext:\n{inp}"
    else:
        user_msg = instruction
        
    formatted_text = (
        f"<start_of_turn>user\n{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n{output}<end_of_turn>"
    )
    return {"text": formatted_text}

formatted_data = [format_record(r) for r in raw_records]

# 3. Create 90/10 train-validation split
random.seed(42)
random.shuffle(formatted_data)

split_idx = int(len(formatted_data) * 0.9)
train_list = formatted_data[:split_idx]
val_list = formatted_data[split_idx:]

dataset = DatasetDict({
    "train": Dataset.from_list(train_list),
    "validation": Dataset.from_list(val_list)
})

print(f"Split Complete: {len(dataset['train'])} Train | {len(dataset['validation'])} Validation")
print("\n--- SAMPLE FORMATTED PROMPT ---")
print(dataset['train'][0]['text'])

Loaded 613 records from tanglish.jsonl
Split Complete: 551 Train | 62 Validation

--- SAMPLE FORMATTED PROMPT ---
<start_of_turn>user
What is a Higher-Order Component (HOC) in React?

Context:
function withLogger(Component) {
  return function Wrapped(props) {
    console.log('Rendering', Component.name);
    return <Component {...props} />;
  };
}<end_of_turn>
<start_of_turn>model
`HOC` oru function, adhu oru component-a eduthukittu, extra functionality-oda oru `new component`-a return pannum. Example: `withAuth`, `withLogger`, Redux-oda `connect()`. Cross-cutting logic-a (auth check, logging) reuse panna idhu use aagum. Aana ippo `custom hooks` vandhadhukku appuram HOCs-oda usage kammi aagirukku, because hooks simple-a and nested wrappers ('wrapper hell') illama irukkum.<end_of_turn>


Load Base Model Gemma-2B and Attach LoRA Adapters

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_id = "google/gemma-2b"

# 1. Tokenizer setup
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 2. Base Model loaded in 4-bit NF4
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

# 3. LoRA Configuration (Rank r=16, Alpha=32)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 4. Wrap model with LoRA
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.
Loading weights: 100%|██████████| 164/164 [00:04<00:00, 33.51it/s]


trainable params: 19,611,648 || all params: 2,525,784,064 || trainable%: 0.7765


Execute QLoRA Fine-Tuning Execution

In [ ]:
import transformers
from trl import SFTTrainer, SFTConfig

# 1. Configuration using SFTConfig
sft_config = SFTConfig(
    output_dir="tanglish_gemma2b_lora_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,  
    warmup_steps=10,               
    num_train_epochs=3,            
    learning_rate=2e-4,
    bf16=True,                     
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=30,
    save_strategy="steps",
    save_steps=60,
    save_total_limit=2,
    optim="paged_adamw_8bit",       
    gradient_checkpointing=True,    
    report_to="none",
    max_length=512
)

def formatting_prompts_func(example):
    return example["text"]

# 2. Initialize SFTTrainer (pass base `model` instead of `peft_model`)
trainer = SFTTrainer(
    model=model,                   
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=lora_config,         
    formatting_func=formatting_prompts_func,
    args=sft_config,
)


/home/prathoseraajv/cross_lingual/.venv/lib64/python3.14/site-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/prathoseraajv/cross_lingual/.venv/lib64/python3.14/site-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Truncating train dataset: 100%|██████████| 551/551 [00:00<00:00, 21834.81 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 551/551 [00:00<00:00, 67228.92 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 62/62 [00:00<00:00, 32566.92 examples/s]


In [15]:
trainer.train()

In file included from /usr/include/python3.14/pyconfig.h:6,
                 from /usr/include/python3.14/Python.h:14,
                 from /home/prathoseraajv/cross_lingual/.venv/lib/python3.14/site-packages/triton/backends/nvidia/driver.c:9:
/usr/include/python3.14/pyconfig-64.h:2031:9: warning: ‘_POSIX_C_SOURCE’ redefined
 2031 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/prathoseraajv/cross_lingual/.venv/lib/python3.14/site-packages/triton/backends/nvidia/include/cuda.h:56,
                 from /home/prathoseraajv/cross_lingual/.venv/lib/python3.14/site-packages/triton/backends/nvidia/driver.c:1:
/usr/include/features.h:319:10: note: this is the location of the previous definition
  319 | # define _POSIX_C_SOURCE        202405L
      |          ^~~~~~~~~~~~~~~


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
30,2.414481,2.427634,2.447055,0.543347,48502.000000
60,2.098116,2.099151,2.067094,0.582835,98938.000000
90,1.757626,1.978436,1.835199,0.598129,147345.000000
120,1.682298,1.936027,1.761042,0.601316,196225.000000
150,1.364200,1.973459,1.468967,0.606098,245827.000000
180,1.412311,1.952248,1.486326,0.605486,295530.000000
207,1.383273,1.938344,1.518023,0.607485,338724.000000


TrainOutput(global_step=207, training_loss=1.9002440171541224, metrics={'train_runtime': 505.6136, 'train_samples_per_second': 3.269, 'train_steps_per_second': 0.409, 'total_flos': 4067728516694016.0, 'train_loss': 1.9002440171541224, 'epoch': 3.0})

In [1]:
# 3. Save trained LoRA adapters
output_adapter_dir = "tanglish_gemma2b_adapter"
trainer.model.save_pretrained(output_adapter_dir)
tokenizer.save_pretrained(output_adapter_dir)

print(f"Adapter weights successfully saved to directory: '{output_adapter_dir}'")

NameError: name 'trainer' is not defined

Testing

In [4]:
text = "<start_of_turn>user\nWhat is computer Science?<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer(text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = trainer.model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

NameError: name 'trainer' is not defined

Using Fintuned Model

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# 1. Define paths
base_model_id = "google/gemma-2b"
adapter_dir = "tanglish_gemma2b_adapter"

# 2. Load Tokenizer & 4-Bit Base Model into RAM
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 3. Load your fine-tuned adapter weights from disk
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

# How to verify it worked: Run a test prompt
text = "<start_of_turn>user\nWhat is computer Science?<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer(text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.
Loading weights: 100%|██████████| 164/164 [00:03<00:00, 49.81it/s]


user
What is computer Science?
model
`Computer Science` oru `discipline` or `field` (subject), not a language. `Problem solving` and `algorithms` (logic) vachi, `data structures` (arrays, lists, trees) and `abstraction` (classes, functions) vachi, `languages` (C, Python) vachi, `hardware` (CPU, memory) vachi, `theory` (probability, complexity) vachi, `applications` (AI, robotics) v


Quantitative Evaluation

In [ ]:
import torch
import math
import numpy as np
from evaluate import load

# 1. Load ROUGE metric
rouge = load("rouge")

val_samples = dataset["validation"]
print(f"Running Quantitative Evaluation on {len(val_samples)} validation records...")

base_losses = []
ft_losses = []
references = []
ft_predictions = []

with torch.no_grad():
    for idx, sample in enumerate(val_samples):
        text = sample["text"]
        
        # Format input prompt and target response
        prompt = text.split("<start_of_turn>model\n")[0] + "<start_of_turn>model\n"
        target_text = text.split("<start_of_turn>model\n")[-1].replace("<end_of_turn>", "").strip()
        
        # Tokenize sequence
        inputs = tokenizer(text, return_tensors="pt").to("cuda")
        labels = inputs["input_ids"].clone()
        
        
        # 1. Calculate Fine-Tuned Model Loss
        ft_outputs = model(input_ids=inputs["input_ids"], labels=labels)
        ft_losses.append(ft_outputs.loss.item())
        
        # 2. Calculate Base Model Loss (unloaded adapter pass)
        with model.disable_adapter():
            base_outputs = model(input_ids=inputs["input_ids"], labels=labels)
            base_losses.append(base_outputs.loss.item())
            
        # 3. Generate response from Fine-Tuned model for ROUGE evaluation
        gen_inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        gen_outputs = model.generate(
            **gen_inputs,
            max_new_tokens=120,
            do_sample=False,  # Greedy search for deterministic metric eval
            eos_token_id=tokenizer.eos_token_id
        )
        
        gen_text = tokenizer.decode(gen_outputs[0], skip_special_tokens=False)
        response_only = gen_text.split("<start_of_turn>model\n")[-1].replace("<end_of_turn>", "").strip()
        
        ft_predictions.append(response_only)
        references.append(target_text)

# 4. Compute Perplexity (PPL = exp(mean_loss))
base_ppl = math.exp(np.mean(base_losses))
ft_ppl = math.exp(np.mean(ft_losses))

# 5. Compute ROUGE Scores
rouge_results = rouge.compute(predictions=ft_predictions, references=references)

# Print Final Research Paper Table for RQ1
print("\n" + "="*60)
print("RQ1 QUANTITATIVE EVALUATION RESULTS")
print("="*60)
print(f"Base Gemma-2B Perplexity (PPL):        {base_ppl:.2f}")
print(f"Fine-Tuned Gemma-2B Perplexity (PPL):   {ft_ppl:.2f}")
print(f"Perplexity Reduction:                  {((base_ppl - ft_ppl) / base_ppl * 100):.2f}%")
print("-" * 60)
print(f"ROUGE-1 Score:                        {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2 Score:                        {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L Score:                        {rouge_results['rougeL']:.4f}")
print("="*60)

Running Quantitative Evaluation on 62 validation records...

RQ1 QUANTITATIVE EVALUATION RESULTS
Base Gemma-2B Perplexity (PPL):        69.70
Fine-Tuned Gemma-2B Perplexity (PPL):   7.27
Perplexity Reduction:                  89.57%
------------------------------------------------------------
ROUGE-1 Score:                        0.3423
ROUGE-2 Score:                        0.0888
ROUGE-L Score:                        0.2153
